In [1]:
!pip -q install -U autoawq transformers accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 5.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 86.2 MB/s eta 0:00:00


In [2]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code = True)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1855: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(


tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [21]:
tokenizer.pad_token = tokenizer.eos_token

In [37]:
import torch
torch.cuda.is_available()

True

In [46]:
model = AutoAWQForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage = True,
    device_map = 'auto',
    use_cache = False,
    trust_remote_code = True,
)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
calib_data = [
    """
    Transformers are deep learning architectures that use self-attention
    mechanisms to process sequential information efficiently.
    Quantization reduces model memory usage and improves inference speed.
    """,

    """
    Large language models are trained on massive datasets using autoregressive
    objectives to predict the next token in a sequence.
    """,

    """
    Activation-aware Weight Quantization preserves important channels
    during low-bit compression of neural network weights.
    """
] * 100

In [14]:
type(calib_data)

list

In [23]:
test = tokenizer(calib_data[0])

print(test.keys())
print(test['input_ids'][:10])

KeysView({'input_ids': [1, 29871, 13, 1678, 4103, 689, 414, 526, 6483, 6509, 6956, 1973, 393, 671, 1583, 29899, 1131, 2509, 13, 1678, 7208, 12903, 304, 1889, 8617, 2556, 2472, 29497, 29889, 13, 1678, 22746, 2133, 26830, 1904, 3370, 8744, 322, 4857, 1960, 27262, 6210, 29889, 13, 268], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})
[1, 29871, 13, 1678, 4103, 689, 414, 526, 6483, 6509]


In [29]:
tokens = tokenizer(
    calib_data,
    padding = True,
    truncation = True,
    return_tensors = 'pt'
)

Why Tokenization?

    Transformer input:

        X ∈ R**(B×T×d)

Need token IDs first.

In [25]:
#awq quantization config
quant_config = {
    "zero_point": True,  #assymetric quant more accurate
    "q_group_size": 128, # one scale shared across 128 weights (diff chaneel have diff distri)
    "w_bit": 4,        # quantize weight to INT4
    'version': 'GEMM'
}

In [35]:
model.quantize(
    tokenizer,
    quant_config=quant_config,
    calib_data=calib_data,
    max_calib_seq_len=256 # Increased from 128 to ensure enough tokens for calibration
)

AWQ: 100%|██████████| 22/22 [06:07<00:00, 16.72s/it]


In [40]:
save_path = "./tinyllama-awq"

model.save_quantized(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards: 0it [00:00, ?it/s]

('./tinyllama-awq/tokenizer_config.json',
 './tinyllama-awq/chat_template.jinja',
 './tinyllama-awq/tokenizer.json')

In [44]:
model_awq_quant = AutoAWQForCausalLM.from_quantized(
    save_path,
    device_map="auto"
)

Replacing layers...: 100%|██████████| 22/22 [00:05<00:00,  3.87it/s]
/usr/local/lib/python3.12/dist-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed.No module named 'awq_ext'
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)


In [54]:
prompt = "Explain transformers in deep learning."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model_awq_quant.generate(
    **inputs,
    max_new_tokens=200,
    top_p=0.7,
    do_sample=True,
    temperature=0.9
)

print(tokenizer.decode(output[0]))

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> Explain transformers in deep learning.
Transformers are the architectural block that transforms CNNs into a more powerful RNN-based language model. They consist of several layers of multi-head self-attention, each containing multiple fully connected layers. The attention module can be used to perform attention over a fixed sequence length, generating a new sequence output using the input sequence as a query. The output is then multiplied by an attention weight matrix to obtain the output sequence.
This transformation allows transformers to process longer sequences in a way that is both efficient and computationally demanding. In deep learning, transformers have been used successfully for NLP tasks, such as machine translation and natural language understanding, which require sequence-to-sequence reasoning.
3. Convolutional Neural Networks (CNNs)
CNNs are a type of deep neural network architecture that is particularly well suited for image recognition tasks. They consist of multiple